In [1]:
!pip install eurostat

In [2]:
import eurostat
df = eurostat.get_data_df('isoc_eb_ai')

In [3]:
df.head()

,freq,size_emp,nace_r2,indic_is,unit,geo\TIME_PERIOD,2021,2023,2024,2025
0,A,0-9,C10-S951_X_K,E_AIX_CC1SIX_DA,PC_ENT,ES,NaN,5.74,5.66,5.49
1,A,0-9,C10-S951_X_K,E_AIX_CC1SIX_DA,PC_ENT,PT,NaN,13.36,NaN,NaN
2,A,0-9,C10-S951_X_K,E_AIX_CC1SIX_DA,PC_ENT_IUSE,ES,NaN,6.88,6.94,6.97
3,A,0-9,C10-S951_X_K,E_AIX_CC1SIX_DA,PC_ENT_IUSE,PT,NaN,15.23,NaN,NaN
4,A,0-9,C10-S951_X_K,E_AIX_CC1SI_DA,PC_ENT,ES,NaN,1.40,1.44,1.95


In [4]:
df.describe()

,2021,2023,2024,2025
count,13290.000000,19871.000000,16931.000000,20664.000000
mean,11.486182,15.178872,14.736223,18.785906
std,18.976019,21.729041,19.197311,21.689725
min,0.000000,0.000000,0.000000,0.000000
25%,1.260000,1.820000,2.745000,4.080000
50%,3.170000,4.630000,6.260000,9.445000
75%,11.977500,19.710000,19.090000,24.880000
max,100.000000,100.000000,100.000000,100.000000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25263 entries, 0 to 25262
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   freq             25263 non-null  object 
 1   size_emp         25263 non-null  object 
 2   nace_r2          25263 non-null  object 
 3   indic_is         25263 non-null  object 
 4   unit             25263 non-null  object 
 5   geo\TIME_PERIOD  25263 non-null  object 
 6   2021             13290 non-null  float64
 7   2023             19871 non-null  float64
 8   2024             16931 non-null  float64
 9   2025             20664 non-null  float64
dtypes: float64(4), object(6)
memory usage: 1.9+ MB


In [6]:
df.shape

(25263, 10)

In [7]:
import pandas as pd

df = df.rename(columns={'geo\\TIME_PERIOD': 'geo'})
df = df.melt(id_vars=['freq', 'size_emp', 'nace_r2', 'indic_is', 'unit', 'geo'],
             var_name='year', value_name='value').dropna(subset=['value'])
df['year'] = df['year'].astype(int)
df.shape

(70756, 8)

In [8]:
usage = df[(df['indic_is'] == 'E_AI_TANY') & (df['unit'] == 'PC_ENT')
           & (df['size_emp'].isin(['10-49', '50-249', 'GE250', 'GE10']))
           & (df['geo'].isin(['PL', 'EU27_2020']))]
usage_table = usage.pivot_table(index=['size_emp', 'year'], columns='geo', values='value')
usage_table

geo            EU27_2020     PL
size_emp year                  
10-49    2021       6.12   1.88
         2023       6.41   2.25
         2024      11.21   3.93
         2025      17.00   5.79
50-249   2021      12.55   4.96
         2023      13.07   6.49
         2024      20.97  10.41
         2025      30.36  14.77
GE10     2021       7.65   2.86
         2023       8.06   3.67
         2024      13.48   5.90
         2025      19.95   8.36
GE250    2021      28.41  17.46
         2023      30.48  24.38
         2024      41.17  32.95
         2025      55.03  45.81

In [9]:
barriers = df[(df['indic_is'].isin(['E_AI_BCST', 'E_AI_BLE', 'E_AI_BINC', 'E_AI_BDDT',
                                    'E_AI_BCDP', 'E_AI_BLEG', 'E_AI_BEC', 'E_AI_BNU']))
              & (df['unit'] == 'PC_ENT_AI_EC') & (df['size_emp'] == '10-49')
              & (df['year'] == 2025) & (df['geo'].isin(['PL', 'EU27_2020']))]
barriers_table = barriers.pivot_table(index='indic_is', columns='geo', values='value')

df.to_csv('eurostat_ai_enterprises.csv', index=False)
usage_table.to_csv('ai_usage_pl_vs_eu.csv')
barriers_table.to_csv('ai_barriers_small_firms_2025.csv')
barriers_table.sort_values('PL', ascending=False)

geo,EU27_2020,PL
indic_is,,
E_AI_BLE,70.87,63.41
E_AI_BCST,38.82,52.88
E_AI_BLEG,54.79,52.82
E_AI_BCDP,52.49,48.34
E_AI_BINC,41.69,39.61
E_AI_BNU,19.22,37.17
E_AI_BDDT,43.66,33.19
E_AI_BEC,25.31,26.89
